# **Sentiment Analysis**

**Task Description**

In this task we will perform sentiment analysis using in-context learning.
For the task, we will use the sentiment of tweets as positive and negative.

## **Importing Necessary Packages**

In [ ]:
!pip install openai --quiet

In [ ]:
import re
import nltk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from wordcloud import WordCloud
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

In [ ]:
np.random.seed(42)

In [ ]:
from openai import OpenAI

OPENAI_KEY = ""
client = OpenAI(api_key=OPENAI_KEY)


## **Data Preprocessing**

### **Exploring the Dataset**

In [ ]:
#Reading data
train_data = pd.read_csv("/kaggle/input/pm-data/train_data_no_dup2.csv")
val_data = pd.read_csv("/kaggle/input/pm-data/test_data_no_dup2.csv")
test_data = pd.read_csv("/kaggle/input/pm-data/test_data_no_dup2.csv")

# Rename columns
train_data.columns = ["NFR1", "NFR2", "conflict status"]
val_data.columns = ["NFR1", "NFR2", "conflict status"]
test_data.columns = ["NFR1", "NFR2", "conflict status"]

train_data.head()

In [ ]:
train_data.shape

In [ ]:
val_data.shape

In [ ]:
test_data.shape

In [ ]:
import warnings

with warnings.catch_warnings():
    warnings.simplefilter("ignore", DeprecationWarning)
    sampled_examples = (
        train_data.groupby('conflict status', group_keys=False)
        .apply(lambda x: x.sample(n=2, random_state=42))
    )
import warnings

warnings.filterwarnings(
    "ignore",
    message=(
        ".*DataFrameGroupBy\\.apply operated on the grouping columns.*"
    ),
    category=DeprecationWarning,
)


## **In-Context Learning**

In [ ]:
prompts = [
    # (1) Keep one of your originals (concise style)
    (
        "You are a requirements analyst. Given two requirements, decide if they are in Conflict or Neutral.\n\n"
        "Answer with one word only: Conflict or Nuetral."
    ),

    # (2) Question-style
    (
        "Do the following two requirements express any kind of conflict or contradiction?\n"
        "If they oppose or cannot be true together, answer 'Conflict'. Otherwise, answer 'Nuetral'.\n\n"
        "Respond with one word: Conflict or Nuetral."
    ),

    # (3) More detailed, semantic reasoning
    (
        "You are analyzing system requirements for logical consistency.\n"
        "Compare the two given requirements carefully — check if their meanings, conditions, or expected outcomes contradict each other.\n"
        "If you find any inconsistency or incompatibility, classify as Conflict; otherwise, classify as Nuetral.\n\n"
        "Answer with a single word: Conflict or Nuetral."
    )
]


In [ ]:
def construct_prompt(test_sample, system_prompt, prompt_template, shot_samples):
    """
    Builds a plain-text few-shot prompt (no chat roles).
    GPT-2-style: system prompt is included as plain text at the top.
    """
    parts = []

    # optional system instruction at the top
    if system_prompt:
        parts.append(system_prompt.strip())
        parts.append("")  # blank line

    # few-shot examples
    for _, sample in shot_samples.iterrows():
        nfr1, nfr2 = sample['NFR1'], sample['NFR2']
        label = "conflict" if sample['conflict status'] == 1 else "no"

        example = (
            prompt_template
            .replace("{{NFR1}}", nfr1)
            .replace("{{NFR2}}", nfr2)
            .replace("{{TARGET}}", label)
        )
        parts.append(example.strip())
        parts.append("")  # blank line between examples

    # final test input (no label yet)
    nfr1, nfr2 = test_sample['NFR1'], test_sample['NFR2']
    test_input = (
        prompt_template
        .replace("{{NFR1}}", nfr1)
        .replace("{{NFR2}}", nfr2)
        .replace("{{TARGET}}", "")
    )
    parts.append(test_input.strip())

    # join into one string prompt
    
    return "\n".join(parts)


In [ ]:

def get_model_prediction(system_prompt, test_sample, shot_samples=pd.DataFrame()):
    prompt_template = (
        "NFR1: '{{NFR1}}'\n"
        "NFR2: '{{NFR2}}'\n"
        "Answer: {{TARGET}}"
    )

    # Build GPT-2-style text prompt (system + few-shot + test)
    text_prompt = construct_prompt(test_sample, system_prompt, prompt_template, shot_samples)
    # print(text_prompt)
    # Call GPT-4 (or gpt-4o-mini, etc.)
    response = client.chat.completions.create(
        model="gpt-4o-mini",  # or "gpt-4o", "gpt-4.1-mini", etc.
        messages=[
            {
                "role": "user",
                "content": text_prompt
            }
        ],
        max_tokens=4,   # enough for "conflict" / "no"
        temperature=0   # deterministic
    )

    # Return raw model output, like before
    return response.choices[0].message.content.strip().lower()


## **Zero-Shot**

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

best=0
best_prompt=None
for i in range(len(prompts)):
    test_predictions = []
    
    # Iterate through both NFR1 and NFR2 together
    for nfr1, nfr2 in zip(test_data['NFR1'], test_data['NFR2']):
        # Build a temporary row-like structure for compatibility with get_model_prediction
        sample = {'NFR1': nfr1, 'NFR2': nfr2}
        # raw = get_model_prediction(prompts[i], sample).lower().strip()
        # m = re.search(r"\b(neutral|conflict)\b", raw, flags=re.IGNORECASE)
        # pred = m.group(1).lower() if m else raw.strip().lower()
        # pred = get_model_prediction(prompts[i], sample).lower()
        raw_pred = get_model_prediction(prompts[i], sample).lower().strip()
        # print(raw_pred)
        if raw_pred.startswith("n"):
            pred = "no"
        elif raw_pred.startswith("c"):
            pred = "conflict"
        else:
            # fallback in case the model outputs something unexpected
            pred = "no"  # or raise an error, depending on your preference
        # print(pred)
        test_predictions.append(pred)
    
    # Convert text predictions to numeric (conflict=1, no_conflict=0)
    test_predictions_num = [1 if p == "conflict" else 0 for p in test_predictions]
        
    # Generate the classification report
    print("\nClassification Report:")
    print(classification_report(test_data['conflict status'], test_predictions_num, target_names=["no", "conflict"], digits=4))
    macro_f1 = f1_score(
        test_data['conflict status'],
        test_predictions_num,
        average='macro'
    )
    
    print("Macro F1-score:", macro_f1)
    if macro_f1> best:
        best= macro_f1
        best_prompt= prompts[i]
    # Generate the confusion matrix
    print("\nConfusion Matrix:")
    cm = confusion_matrix(test_data['conflict status'], test_predictions_num)
    
    # Plot the confusion matrix
    plt.figure(figsize=(7, 7))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", square=True,
                xticklabels=["neutral", "conflict"],
                yticklabels=["neutral", "conflict"])
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.title('Confusion Matrix')
    plt.show()

In [ ]:
print(best_prompt)

## **Two-Shots**

In [ ]:

best=0
best_prompt=None
for i in range(len(prompts)):
    test_predictions = []

    # Loop through all NFR pairs in your testidation data
    for nfr1, nfr2 in zip(test_data['NFR1'], test_data['NFR2']):
        # Randomly sample a few examples from the training data for few-shot context
        sampled_examples = train_data.sample(n=2)
        # print(sampled_examples)
        # print(sampled_examples)

    
        # Build the sample dict expected by get_model_prediction()
        sample = {'NFR1': nfr1, 'NFR2': nfr2}
    
        # Get model prediction
        # raw = get_model_prediction(prompts[i], sample, sampled_examples).lower().strip()
        # m = re.search(r"\b(neutral|conflict)\b", raw, flags=re.IGNORECASE)
        # pred = m.group(1).lower() if m else raw.strip().lower()
        pred = get_model_prediction(prompts[i], sample, sampled_examples).lower()
        raw_pred = get_model_prediction(prompts[i], sample, sampled_examples).lower().strip()
        # print(raw_pred)
        if raw_pred.startswith("n"):
            pred = "no"
        elif raw_pred.startswith("c"):
            pred = "conflict"
        else:
            # fallback in case the model outputs something unexpected
            pred = "no"  # or raise an error, depending on your preference
        # print(pred)
        # Store it
        test_predictions.append(pred)
        
    # Convert GPT text outputs ("conflict"/"no_conflict") to numeric
    test_predictions_num = [1 if p == "conflict" else 0 for p in test_predictions]
    
    # Generate the classification report
    print("\nClassification Report:")
    print(classification_report(test_data['conflict status'], test_predictions_num,
                                target_names=["neutral (0)", "conflict (1)"], digits=4))
    macro_f1 = f1_score(
            test_data['conflict status'],
            test_predictions_num,
            average='macro'
        )
        
    print("Macro F1-score:", macro_f1)
    if macro_f1> best:
        best= macro_f1
        best_prompt= prompts[i]
    # Generate the confusion matrix
    print("\nConfusion Matrix:")
    cm = confusion_matrix(test_data['conflict status'], test_predictions_num)
    
    # Plot the confusion matrix
    plt.figure(figsize=(7, 7))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", square=True,
                xticklabels=["neutral (0)", "conflict (1)"],
                yticklabels=["neutral (0)", "conflict (1)"])
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.title('Confusion Matrix')
    plt.show()


In [ ]:
print(best_prompt)

## **Four-Shots**

In [ ]:

best=0
best_prompt=None
for i in range(len(prompts)):
    test_predictions = []

    # Loop through all NFR pairs in your testidation data
    for nfr1, nfr2 in zip(test_data['NFR1'], test_data['NFR2']):
        # Randomly sample a few examples from the training data for few-shot context
        sampled_examples = train_data.sample(n=4)
        # print(sampled_examples)
        # print(sampled_examples)

    
        # Build the sample dict expected by get_model_prediction()
        sample = {'NFR1': nfr1, 'NFR2': nfr2}
    
        # Get model prediction
        # raw = get_model_prediction(prompts[i], sample, sampled_examples).lower().strip()
        # m = re.search(r"\b(neutral|conflict)\b", raw, flags=re.IGNORECASE)
        # pred = m.group(1).lower() if m else raw.strip().lower()
        pred = get_model_prediction(prompts[i], sample, sampled_examples).lower()
        raw_pred = get_model_prediction(prompts[i], sample, sampled_examples).lower().strip()

        if raw_pred.startswith("n"):
            pred = "no"
        elif raw_pred.startswith("c"):
            pred = "conflict"
        else:
            # fallback in case the model outputs something unexpected
            pred = "no"  # or raise an error, depending on your preference
        # print(pred)
        # Store it
        test_predictions.append(pred)
        
    # Convert GPT text outputs ("conflict"/"no_conflict") to numeric
    test_predictions_num = [1 if p == "conflict" else 0 for p in test_predictions]
    
    # Generate the classification report
    print("\nClassification Report:")
    print(classification_report(test_data['conflict status'], test_predictions_num,
                                target_names=["neutral (0)", "conflict (1)"], digits=4))
    macro_f1 = f1_score(
            test_data['conflict status'],
            test_predictions_num,
            average='macro'
        )
        
    print("Macro F1-score:", macro_f1)
    if macro_f1> best:
        best= macro_f1
        best_prompt= prompts[i]
    # Generate the confusion matrix
    print("\nConfusion Matrix:")
    cm = confusion_matrix(test_data['conflict status'], test_predictions_num)
    
    # Plot the confusion matrix
    plt.figure(figsize=(7, 7))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", square=True,
                xticklabels=["neutral (0)", "conflict (1)"],
                yticklabels=["neutral (0)", "conflict (1)"])
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.title('Confusion Matrix')
    plt.show()


In [ ]:
print(best_prompt)